## Stage 3 - Dino Contrastive

### Data folder structure 

In [1]:
import os

for root, dirs, files in os.walk("JIGSAWS"):
    level = root.replace("JIGSAWS", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for f in files[:3]:
            print(f"    {indent}{f}")

JIGSAWS/
    .DS_Store
  Experimental_setup/
      .DS_Store
    Knot_Tying/
        .DS_Store
      Balanced/
        GestureClassification/
          OneTrialOut/
            25_Out/
              itr_44/
              itr_43/
              itr_17/
              itr_28/
              itr_10/
              itr_9/
              itr_26/
              itr_19/
              itr_21/
              itr_7/
              itr_42/
              itr_45/
              itr_6/
              itr_20/
              itr_27/
              itr_1/
              itr_18/
              itr_11/
              itr_8/
              itr_16/
              itr_29/
              itr_34/
              itr_33/
              itr_32/
              itr_35/
              itr_50/
              itr_13/
              itr_14/
              itr_22/
              itr_4/
              itr_3/
              itr_25/
              itr_49/
              itr_40/
              itr_47/
              itr_24/
              itr_2/
         

In [2]:
print(os.listdir("Cholec80"))
print(os.listdir("JIGSAWS"))

['.DS_Store', 'cholec80']
['.DS_Store', 'Experimental_setup', 'Knot_Tying', 'Suturing', 'Needle_Passing']


In [3]:
with open("JIGSAWS/Suturing/transcriptions/Suturing_B001.txt") as f:
    print(f.read())

80 219 G1 
220 370 G5 
371 590 G8 
591 660 G2 
661 954 G3 
955 1097 G8 
1098 1124 G2 
1125 1401 G3 
1402 1439 G2 
1440 1698 G8 
1699 1783 G2 
1784 2285 G3 
2286 2495 G6 
2496 2679 G4 
2680 2750 G2 
2751 2976 G3 
2977 3155 G6 
3156 3308 G4 
3309 3659 G2 
3660 4011 G3 
4012 4321 G8 
4322 4374 G2 
4375 4704 G3 
4705 4846 G6 
4847 4983 G4 
4984 5059 G2 
5060 5354 G3 
5355 5447 G6 
5448 5635 G11 



In [4]:
videos = os.listdir("JIGSAWS/Suturing/video/")
print(sorted(videos)[:5])
print(f"Total videos: {len(videos)}")

['Suturing_B001_capture1.avi', 'Suturing_B001_capture2.avi', 'Suturing_B002_capture1.avi', 'Suturing_B002_capture2.avi', 'Suturing_B003_capture1.avi']
Total videos: 78


### Dataloader

In [5]:
import torch 
from torch.utils.data import Dataset, DataLoader
import av
import numpy as np 
import os
from torchvision import transforms
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ensures deterministic behavior in multi-head attention
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# mapping gestures to numbers
GESTURE_MAP = {
    'G1':0,'G2':1,'G3':2,'G4':3,'G5':4,
    'G6':5,'G8':6,'G9':7,'G10':8,'G11':9
}

In [6]:
class JIGSAWSClipDataset(Dataset):
    def __init__(self, task='Suturing', split_trials=None, clip_len=8):
        # split_trials is the list of trial names to include like 'B001', 'B002'

        self.clip_len = clip_len
        self.clips = [] # will (store video_path, start, end, label)

        trans_dir = f"JIGSAWS/{task}/transcriptions/"
        video_dir = f"JIGSAWS/{task}/video/"

        # looping through every transcription file - one per video
        for fname in sorted(os.listdir(trans_dir)):
            if not fname.endswith('.txt'):
                continue

            # extract trial name
            trial = fname.replace(f'{task}_', '').replace('.txt', '')

            # test trials skipped
            if split_trials and trial not in split_trials:
                continue
            
            # we use capture 1 camera angle
            video_path = os.path.join(video_dir, f"{task}_{trial}_capture1.avi") 
            if not os.path.exists(video_path):
                continue

            # read transcription file
            with open(os.path.join(trans_dir, fname)) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 3: continue
                    start, end, gesture = int(parts[0]), int(parts[1]), parts[2]

                    # keeping only gestures we know and long enough to sample 8 frames
                    if gesture in GESTURE_MAP and (end-start)>clip_len:
                        self.clips.append((video_path, start, end, GESTURE_MAP[gesture]))

                
        # transform to match ResNet
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((224, 224), antialias = True),
            transforms.Normalize([0.485,0.456,0.406],
                                [0.229,0.224,0.225]) 
        ])
        print(f"    {task} {split_trials}: {len(self.clips)} gesture clips")

    
    def _load_frames(self, video_path, start, end):
        # pick clip_len frame indices evenly spread across the gesture segment

        frame_ids = set(np.linspace(start, end, self.clip_len, dtype=int).tolist())
        frames = []
        container = av.open(video_path) # open the video file
        for i, frame in enumerate(container.decode(video=0)):
            if i in frame_ids:
                img = frame.to_ndarray(format='rgb24') # converting to RGB numpy array
                frames.append(self.transform(img)) # resize + transform
            if len(frames) == self.clip_len:
                break # stop when we have all frames

        container.close()

        while len(frames) < self.clip_len:
            frames.append(frames[-1]) # pad the last frame if less

        return torch.stack(frames) # stack into (T, C, H, W) tensor

    def __len__(self): return len(self.clips)

    def __getitem__(self,idx):
        video_path, start, end, label = self.clips[idx]
        frames = self._load_frames(video_path, start, end)

        # for stage 2 we return all frames
        return frames, label            

In [7]:
# loading a sample to check

ds = JIGSAWSClipDataset(task="Suturing")
img, label = ds[0]
print(f"Sample shape: {img.shape}, label: {label}, gesture: {list(GESTURE_MAP.keys())[label]}")

    Suturing None: 793 gesture clips
Sample shape: torch.Size([8, 3, 224, 224]), label: 0, gesture: G1


### Train/test split

In [8]:
# get all trial names from the transciptions folder
all_trials = [f.replace('Suturing_', '').replace('.txt','')
            for f in os.listdir('JIGSAWS/Suturing/transcriptions/')
            if f.endswith('.txt')]
all_trials = sorted(all_trials)
print("All trials:", all_trials)

# first four trials for test rest train
# JIGSAWS uses leave-one-user-out i am doing it in later stages

test_trials = all_trials[:4]
train_trials = all_trials[4:]

print(f"Train: {train_trials}")
print(f"Test: {test_trials}")

#  create dataset objects
train_clip_ds = JIGSAWSClipDataset('Suturing', split_trials=train_trials)
test_clip_ds  = JIGSAWSClipDataset('Suturing', split_trials=test_trials)

# dataloadaer handles batching + shuffling
train_clip_loader = DataLoader(train_clip_ds, batch_size=16, shuffle=True, num_workers=0)
test_clip_loader = DataLoader(test_clip_ds, batch_size=16, shuffle=False, num_workers=0)
print(f"\nTrain batches: {len(train_clip_loader)}, Test batches: {len(test_clip_loader)}")

All trials: ['B001', 'B002', 'B003', 'B004', 'B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']
Train: ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']
Test: ['B001', 'B002', 'B003', 'B004']
    Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 gesture clips
    Suturing ['B001', 'B002', 'B003

### DINO-VIT Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import timm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")


class DinoSurgicalTranformer(nn.Module):
    def __init__(self, n_classes=10, clip_len=8, feat_dim=384,
                 proj_dim=256, n_heads=6, n_layers=4, dropout=0.1):
        super().__init__()
        self.clip_len = clip_len
        
        # DINO ViT - ouputs 384 dim features per frame

        self.vit = timm.create_model(
            'vit_small_patch16_224.dino',
            pretrained=True,
            num_classes=0  # we remove classification head 
            # out 384-dim features
        )

        for param in self.vit.parameters():
            param.requires_grad = False # freeze all ViT weights

        
        # Temporal Transformer same as stage 2

        self.cls_token = nn.Parameter(torch.randn(1, 1, feat_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, clip_len+1, feat_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model = feat_dim, nhead=n_heads,
            dim_feedforward=feat_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.norm = nn.LayerNorm(feat_dim)

        # adding projection head
        # projets clip embedding to lower dim space for contranstive learning

        self.proj_head = nn.Sequential(
            nn.Linear(feat_dim, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

        # prototype classifier - one learnable vector per gesture, training pulls same class embeddings toward their prototype
        self.prototypes = nn.Parameter(torch.randn(n_classes, proj_dim))

        self.temperature = 0.07  # InfoNCE temp - controls how sharp the distribution is

    def forward(self, x, labels=None):
        # x: (B, T, C, H, W)
        # labels: (B, ) - gesture labels needed during training for InfoNCE loss
        # returns loss + logits during training, only logits during inference

        B, T, C, H, W = x.shape

        # frozen DINO-ViT extracts features from every frame
        x = x.view(B*T, C, H, W)  # every frame is independant images
        with torch.no_grad():
            x = self.vit(x)     # (B*T, 384)
        x = x.view(B, T, -1)    # (B, T, 384)


        # prepend the CLS token and add positional embeds

        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, T+1, 384)
        x = x + self.pos_embed


        # temporal self attention across all frames
        x = self.transformer(x)
        x = self.norm(x)

        # clip level summary
        clip_repr = x[:, 0, :]  #(B, 384)

        # project into contrastive embed space
        h = self.proj_head(clip_repr)   #(B, proj_head)

        # cosine similarity to all prototypes
        h_norm = F.normalize(h, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        logits = h_norm @ p_norm.T / self.temperature   # (B, n_classes)

        if labels is not None:
            # InfoNCE loss is cross entropy on cosine sim logits
            # this pulls h close to correct proto and pushes away from others

            loss = F.cross_entropy(logits, labels)
            return loss, logits
        return logits
    

model3 = DinoSurgicalTranformer(n_classes=10, clip_len=8).to(device)

# train temporal + proj head + proto as ViT is frozen
trainable = [p for p in model3.parameters() if p.requires_grad]
optimizer3 = torch.optim.Adam(trainable, lr=1e-4, weight_decay=1e-4)
scheduler3 = torch.optim.lr_scheduler.StepLR(optimizer3, step_size=5, gamma=0.5)

total = sum(p.numel() for p in model3.parameters() if p.requires_grad)
print(f"Trainable prams: {total:,} (frozen ViT not counted)")
print(f"Device: {device}")


Trainable prams: 7,269,376 (frozen ViT not counted)
Device: mps


### Suturing dataloader

In [12]:
train_ds3 = JIGSAWSClipDataset('Suturing', split_trials=train_trials)
test_ds3 = JIGSAWSClipDataset('Suturing', split_trials=test_trials)

train_loader3 = DataLoader(train_ds3, batch_size=16, shuffle=True, num_workers=0)
test_loader3 = DataLoader(test_ds3, batch_size=16, shuffle=False, num_workers=0)

print(f"Train clips: {len(train_ds3)}, Test clips: {len(test_ds3)}")
print(f"Train batches: {len(train_loader3)}, Test batches: {len(test_loader3)}")

    Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 gesture clips
    Suturing ['B001', 'B002', 'B003', 'B004']: 83 gesture clips
Train clips: 710, Test clips: 83
Train batches: 45, Test batches: 6


### Training loop

In [13]:
from sklearn.metrics import f1_score
import time

print("Stage 3 - DINO-ViT + Contranstive learning (Suturing)")
print("="*50)

best_test_f1 = 0

for epoch in range(1,16):
    t0 = time.time()

    # train
    model3.train()
    tr_preds, tr_labels_list = [], []
    for clips, labels in train_loader3:
        clips, labels = clips.to(device), labels.to(device)

        # model returns loss + logits during train
        loss, logits  = model3(clips, labels=labels)

        optimizer3.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
        optimizer3.step()

        tr_preds.extend(logits.argmax(1).cpu().numpy())
        tr_labels_list.extend(labels.cpu().numpy())

    scheduler3.step()
    tr_f1 = f1_score(tr_labels_list, tr_preds, average="macro", zero_division=0)

    # eval
    te_preds, te_labels_list = [], []

    with torch.no_grad():
        for clips, labels in test_loader3:
            clips, lables = clips.to(device), labels.to(device)

            # no labels at inference and returns on logits
            logits = model3(clips)

            te_preds.extend(logits.argmax(1).cpu().numpy())
            te_labels_list.extend(labels.cpu().numpy())

        te_f1 = f1_score(te_labels_list, te_preds, average="macro", zero_division=0)

        if te_f1 > best_test_f1:
            best_test_f1 = te_f1

    print(f"Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | Test F1: {te_f1:.3f} | {time.time()-t0:.1f}s")
    
print(f"\nStage 3 Best: {best_test_f1:.3f}")



Stage 3 - DINO-ViT + Contranstive learning (Suturing)
Epoch  1 | Train F1: 0.104 | Test F1: 0.150 | 101.9s
Epoch  2 | Train F1: 0.345 | Test F1: 0.473 | 97.2s
Epoch  3 | Train F1: 0.551 | Test F1: 0.668 | 97.2s
Epoch  4 | Train F1: 0.698 | Test F1: 0.741 | 97.8s
Epoch  5 | Train F1: 0.766 | Test F1: 0.700 | 97.1s
Epoch  6 | Train F1: 0.821 | Test F1: 0.864 | 97.1s
Epoch  7 | Train F1: 0.841 | Test F1: 0.730 | 97.2s
Epoch  8 | Train F1: 0.856 | Test F1: 0.829 | 97.0s
Epoch  9 | Train F1: 0.872 | Test F1: 0.804 | 97.4s
Epoch 10 | Train F1: 0.882 | Test F1: 0.815 | 98.0s
Epoch 11 | Train F1: 0.883 | Test F1: 0.880 | 97.6s
Epoch 12 | Train F1: 0.886 | Test F1: 0.853 | 97.8s
Epoch 13 | Train F1: 0.888 | Test F1: 0.839 | 97.4s
Epoch 14 | Train F1: 0.890 | Test F1: 0.894 | 96.9s
Epoch 15 | Train F1: 0.890 | Test F1: 0.830 | 96.9s

Stage 3 Best: 0.894


In [ ]:
task = ['Knot_Tying', 'Needle_Passing']
task3_results = {}

for task in task:
    print(f"\n--- {task} ---")

    task_train_ds = JIGSAWSClipDataset(task, split_trials=train_trials)
    task_test_ds = JIGSAWSClipDataset(task, split_trials=test_trials)

    if len(task_test_ds) == 0:
        print("Skipping - no test clips")
        continue
    task_train_loader = DataLoader(task_train_ds, batch_size=16, shuffle=True, num_workers=0)
    task_test_loader = DataLoader(task_test_ds, batch_size=16, shuffle=True, num_workers=0)

    torch.manual_seed(SEED)
    tm = DinoSurgicalTranformer(n_classes=10, clip_len=8).to(device)
    trainable_m = [p for p in tm.parameters() if p.requires_grad]
    to = torch.optim.Adam(trainable_m, lr=1e-4, weight_decay=1e-4)
    ts = torch.optim.lr_scheduler.StepLR(to, step_size=5, gamma=0.5)

    best_f1 = 0

    for epoch in range(1,16):
        # train

        tm.train()
        tr_preds, tr_labels_list = [], []
        for clips, labels in task_train_loader:
            clips, labels = clips.to(device), labels.to(device)
            loss, logits = tm(clips, labels=labels)
            to.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_m, max_norm=1.0)
            to.step()
            tr_preds.extend(logits.argmax(1).cpu().numpy())
            tr_labels_list.extend(labels.cpu().numpy())
        ts.step()
        tr_f1 = f1_score(tr_labels_list, tr_preds, average="macro", zero_division=0)

        # eval
        tm.eval()
        te_preds, te_labels_list = [], []
        with torch.no_grad():
            for clips, labels in task_test_loader:
                clips, labels = clips.to(device), labels.to(device)
                logits = tm(clips)
                te_preds.extend(logits.argmax(1).cpu().numpy())
                te_labels_list.extend(labels.cpu().numpy())
        te_f1 = f1_score(te_labels_list, te_preds, average="macro", zero_division=0)


        if te_f1 > best_f1:
            best_f1 = te_f1
        
        print(f"  Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | Test F1: {te_f1:.3f}")

    
    task3_results[task] = best_f1


# summary
print("\n"+"="*50)
for task, f1 in task3_results.items():
    print(f" {task:<20}: Best Test F1 = {f1:.3f}")


--- Knot_Tying ---
    Knot_Tying ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 51 gesture clips
    Knot_Tying ['B001', 'B002', 'B003', 'B004']: 4 gesture clips
  Epoch  1 | Train F1: 0.162 | Test F1: 1.000
  Epoch  2 | Train F1: 0.386 | Test F1: 1.000
  Epoch  3 | Train F1: 0.386 | Test F1: 1.000
  Epoch  4 | Train F1: 0.912 | Test F1: 1.000
  Epoch  5 | Train F1: 1.000 | Test F1: 1.000
  Epoch  6 | Train F1: 1.000 | Test F1: 1.000
  Epoch  7 | Train F1: 1.000 | Test F1: 1.000
  Epoch  8 | Train F1: 1.000 | Test F1: 1.000
  Epoch  9 | Train F1: 1.000 | Test F1: 1.000
  Epoch 10 | Train F1: 1.000 | Test F1: 1.000
  Epoch 11 | Train F1: 1.000 | Test F1: 1.000
  Epoch 12 | Train F1: 1.000 | Test F1: 1.000
  Epoch 13 | Train F1: 1.000 | Test F1: 1.000
  E

In [15]:
# continue training Needle_Passing for more epochs
print("Needle_Passing — extended training")
print("="*45)

# reload best needle passing model from the loop above
# since tm was the last task in the loop it's still in memory
best_f1_np = task3_results['Needle_Passing']

for epoch in range(16, 31):
    # train
    tm.train()
    tr_preds, tr_labels_list = [], []
    for clips, labels in task_train_loader:
        clips, labels = clips.to(device), labels.to(device)
        loss, logits = tm(clips, labels=labels)
        to.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_m, max_norm=1.0)
        to.step()
        tr_preds.extend(logits.argmax(1).cpu().numpy())
        tr_labels_list.extend(labels.cpu().numpy())
    ts.step()
    tr_f1 = f1_score(tr_labels_list, tr_preds, average='macro', zero_division=0)

    # eval
    tm.eval()
    te_preds, te_labels_list = [], []
    with torch.no_grad():
        for clips, labels in task_test_loader:
            clips, labels = clips.to(device), labels.to(device)
            logits = tm(clips)
            te_preds.extend(logits.argmax(1).cpu().numpy())
            te_labels_list.extend(labels.cpu().numpy())
    te_f1 = f1_score(te_labels_list, te_preds, average='macro', zero_division=0)

    if te_f1 > best_f1_np:
        best_f1_np = te_f1

    print(f"  Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | Test F1: {te_f1:.3f}")

print(f"\nNeedle_Passing extended best: {best_f1_np:.3f}")
print(f"Previous best (15 epochs):    {task3_results['Needle_Passing']:.3f}")

Needle_Passing — extended training
  Epoch 16 | Train F1: 0.740 | Test F1: 0.527
  Epoch 17 | Train F1: 0.740 | Test F1: 0.563
  Epoch 18 | Train F1: 0.741 | Test F1: 0.531
  Epoch 19 | Train F1: 0.746 | Test F1: 0.532
  Epoch 20 | Train F1: 0.746 | Test F1: 0.529
  Epoch 21 | Train F1: 0.754 | Test F1: 0.555
  Epoch 22 | Train F1: 0.745 | Test F1: 0.567
  Epoch 23 | Train F1: 0.748 | Test F1: 0.555
  Epoch 24 | Train F1: 0.747 | Test F1: 0.569
  Epoch 25 | Train F1: 0.746 | Test F1: 0.518
  Epoch 26 | Train F1: 0.755 | Test F1: 0.567
  Epoch 27 | Train F1: 0.752 | Test F1: 0.567
  Epoch 28 | Train F1: 0.750 | Test F1: 0.552
  Epoch 29 | Train F1: 0.751 | Test F1: 0.567
  Epoch 30 | Train F1: 0.751 | Test F1: 0.569

Needle_Passing extended best: 0.604
Previous best (15 epochs):    0.604


In [10]:
# Stage 3 — Combined
from torch.utils.data import ConcatDataset
from sklearn.metrics import f1_score
import time

print("Stage 3 - Combined (all 3 tasks)")
print("="*55)

# combined dataloaders
tasks = ['Suturing', 'Knot_Tying', 'Needle_Passing']
train_ds3_combined = ConcatDataset([JIGSAWSClipDataset(t, split_trials=train_trials) for t in tasks])
test_ds3_combined  = ConcatDataset([JIGSAWSClipDataset(t, split_trials=test_trials)  for t in tasks])

train_loader3_combined = DataLoader(train_ds3_combined, batch_size=16, shuffle=True,  num_workers=0)
test_loader3_combined  = DataLoader(test_ds3_combined,  batch_size=16, shuffle=False, num_workers=0)

print(f"Total train clips: {len(train_ds3_combined)}")
print(f"Total test clips:  {len(test_ds3_combined)}")

# fresh model
torch.manual_seed(SEED)
model3_combined = DinoSurgicalTranformer(n_classes=10, clip_len=8).to(device)
trainable_combined = [p for p in model3_combined.parameters() if p.requires_grad]
optimizer3_combined = torch.optim.Adam(trainable_combined, lr=1e-4, weight_decay=1e-4)
scheduler3_combined = torch.optim.lr_scheduler.StepLR(optimizer3_combined, step_size=5, gamma=0.5)

best_combined_f1 = 0

for epoch in range(1, 16):
    t0 = time.time()

    # train
    model3_combined.train()
    tr_preds, tr_labels_list = [], []
    for clips, labels in train_loader3_combined:
        clips, labels = clips.to(device), labels.to(device)
        loss, logits = model3_combined(clips, labels=labels)
        optimizer3_combined.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_combined, max_norm=1.0)
        optimizer3_combined.step()
        tr_preds.extend(logits.argmax(1).cpu().numpy())
        tr_labels_list.extend(labels.cpu().numpy())
    scheduler3_combined.step()
    tr_f1 = f1_score(tr_labels_list, tr_preds, average='macro', zero_division=0)

    # eval
    model3_combined.eval()
    te_preds, te_labels_list = [], []
    with torch.no_grad():
        for clips, labels in test_loader3_combined:
            clips, labels = clips.to(device), labels.to(device)
            logits = model3_combined(clips)
            te_preds.extend(logits.argmax(1).cpu().numpy())
            te_labels_list.extend(labels.cpu().numpy())
    te_f1 = f1_score(te_labels_list, te_preds, average='macro', zero_division=0)

    if te_f1 > best_combined_f1:
        best_combined_f1 = te_f1
        # saving best combined model checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model3_combined.state_dict(),
            'optimizer_state_dict': optimizer3_combined.state_dict(),
            'best_f1': best_combined_f1,
        }, 'stage3_combined_best.pt')

    print(f"Epoch {epoch:2d} | Train F1: {tr_f1:.3f} | Test F1: {te_f1:.3f} | {time.time()-t0:.1f}s")

print(f"\nStage 3 Combined Best Test F1: {best_combined_f1:.3f}")

Stage 3 - Combined (all 3 tasks)
    Suturing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 710 gesture clips
    Knot_Tying ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 51 gesture clips
    Needle_Passing ['B005', 'C001', 'C002', 'C003', 'C004', 'C005', 'D001', 'D002', 'D003', 'D004', 'D005', 'E001', 'E002', 'E003', 'E004', 'E005', 'F001', 'F002', 'F003', 'F004', 'F005', 'G001', 'G002', 'G003', 'G004', 'G005', 'H001', 'H003', 'H004', 'H005', 'I001', 'I002', 'I003', 'I004', 'I005']: 453 gesture clips
    Suturing ['B001',